# TE04 - Binary Image Post-Processing and Connected Components

## Reference Links

- OpenCV docs: http://docs.opencv.org/3.2.0/
- OpenCV morphology tutorial: https://docs.opencv.org/3.2.0/d9/d61/tutorial_py_morphological_ops.html
- OpenCV contours tutorial: https://docs.opencv.org/3.2.0/d4/d73/tutorial_py_contours_begin.html
- NumPy docs: https://numpy.org/doc/stable/
- Matplotlib docs: https://matplotlib.org/stable/

## 0. Setup

Expected / optional files for this TE:
- `../data/te04/code_postal.png`
- `../data/te04/passagepieton.jpg`
- `../data/te04/Film_AVI_1.avi`
- `../data/te04/Film_AVI_2.avi`

Fallback already present in the repository:
- `../data/te03/voilier_oies_blanches.jpg`
- `../data/te02/voilier_oies_blanches.jpg`
- `../data/te01/voilier_oies_blanches.jpg`

The setup below creates the output folder and helps you locate available files without failing immediately when optional TE04 data is missing.

## Morphology Primer

- Mathematical morphology studies shapes through the interaction between a binary image and a structuring element.
- Erosion removes foreground pixels that cannot fully contain the structuring element, so thin parts and isolated details tend to disappear first.
- Dilation expands the foreground, which helps connect nearby regions and fill small gaps.
- Morphological opening smooths small bright structures and removes tiny foreground noise, while morphological closing fills narrow holes and reconnects fragmented regions.
- In practice, the effect depends strongly on both the size and the shape of the structuring element.

Source: `SegmentationMorphoMath_SE_SOIA_ROB_2122.pdf`


## 1. Binary Image Post-Processing with Mathematical Morphology

Goal from the statement:
- evaluate the effect of erosion, dilation, opening, closing, and opening-closing on a binary image
- start with a rectangular `3 x 3` structuring element
- compare what you observe with the expected theoretical behavior

### 1.1 Binarization of a source image

Tasks:
1. Choose a binary-processing source image, for example `code_postal.png` if available.
2. Otherwise, reuse `voilier_oies_blanches.jpg` and convert it to grayscale first.
3. Produce a binary image with values in `{0, 255}`.
4. Display both the grayscale image and the binary result.

### 1.2 Erosion, dilation, opening, closing, opening-closing

Use the binary image from 1.1 and a rectangular `3 x 3` structuring element.

Tasks:
- erode the binary image
- dilate the binary image
- apply opening
- apply closing
- apply opening followed by closing
- compare all results visually

### 1.3 Interpretation answers

- The observed results match the theory: erosion shrinks the white foreground, while dilation expands it.
- Thin structures and isolated blobs disappear first under erosion because the structuring element no longer fits inside them.
- Dilation and especially closing reduce small gaps and holes by reconnecting nearby foreground pixels.
- Opening-closing is useful here because it removes small noise first and then reconnects cleaner structures.


### 1.4 Change the structuring element

Tasks:
1. Repeat the experiment with different sizes such as `5 x 5` and `7 x 7`.
2. Compare at least three shapes: rectangle, ellipse, cross.
3. Comment on the sensitivity of the result to the structuring element.

### 1.5 Re-implement with `cv2.morphologyEx`

Task:
- reproduce the previous processing pipeline using `cv2.morphologyEx` with the appropriate flags
- verify that the results match the versions built from `cv2.erode` and `cv2.dilate`

## 2. Extraction and Display of Connected Components

Reminder from the statement:
- thresholding / binarization is only the first part of segmentation
- connected-component labeling is the second part
- the target binary classes here are background (`0`) and birds (`1` or `255`)

### Connected Components Theory

- Thresholding produces a binary partition, but connected-component labeling turns that partition into distinct regions with their own identity.
- A connected component is a maximal set of foreground pixels linked under a chosen neighborhood system.
- 4-connectivity uses horizontal and vertical adjacency only, while 8-connectivity also links diagonal contacts; the choice can change the counted regions.
- Once regions are labeled, properties such as area, centroid, and bounding box become available for filtering and interpretation.

Source: `SegmentationMorphoMath_SE_SOIA_ROB_2122.pdf`


### 2.1 Binarize `voilier_oies_blanches.jpg` into background / birds

Tasks:
1. Load `voilier_oies_blanches.jpg` in grayscale.
2. Find a threshold that separates the sky / background from the birds.
3. Produce a clean binary image where birds are foreground.
4. If necessary, add a light morphology post-processing step before labeling.

### 2.2 Label connected components

Tasks:
- use `cv2.connectedComponents` or `cv2.connectedComponentsWithStats`
- optionally compare with `scipy.ndimage.label` if SciPy is available in your environment
- visualize the label image with a colormap
- inspect how many connected components are detected

### 2.3 Analysis questions

Answer after testing:
- Does the number of detected connected components look correct?
- If not, is the issue caused by thresholding, touching objects, noise, holes, or connectivity choice?
- Would `4`-connectivity and `8`-connectivity give the same result here? Why?

### 2.4 Alternative approach with `cv2.findContours`

Tasks from the statement:
1. Read the documentation for `cv2.findContours` and explain in a few words what it returns.
2. Extract external and internal contours from the binary image.
3. Draw them on the color version of `voilier_oies_blanches.jpg`.
4. Display external contours in yellow.
5. Add a visualization constraint so that the inside of connected components appears in cyan.

### Contour Theory Note

- A contour is the boundary that separates adjacent regions with different image properties.
- Connected components describe full foreground regions, whereas contours describe their borders and can also encode hierarchy between outer boundaries and holes.
- On binary objects, contour extraction is effective because the segmentation step has already isolated the foreground from the background.
- In this TE, contour visualization is useful because it shows both object localization and region structure without replacing component labeling.

Source: `Contours_SE_SOIA_ROB_TNI_2223.pdf`, `SegmentationMorphoMath_SE_SOIA_ROB_2122.pdf`


## 3. Real-Time Adaptation to Webcam or AVI Input

Tasks:
1. Adapt your contour-based pipeline to a grayscale webcam stream.
2. Point the camera toward a simple object placed on a uniform background.
3. If no webcam is available, use one of the AVI files mentioned in the statement.
4. Save the segmented sequence.
5. Describe and criticize the observed behavior.

### Contours in Video and Counting Pipelines

- In live segmentation, contours provide a compact representation of detected objects that can be overlaid on the original frames.
- Their usefulness depends on contour stability: noise, illumination changes, and weak boundaries can create broken or misleading borders.
- For counting tasks, contours and connected components are complementary: one offers boundary geometry, the other offers region-level measurements.
- This is why the crosswalk challenge combines thresholding, morphology, and region filtering instead of relying on one representation only.

Source: `Contours_SE_SOIA_ROB_TNI_2223.pdf`


## 4. Challenge - Count the White Stripes of a Crosswalk

Statement goal:
- automatically count the number of white stripes in `passagepieton.jpg`

Before coding, propose a processing pipeline on paper or in markdown:
1. preprocessing
2. thresholding / segmentation
3. morphology cleanup
4. connected components or contour filtering
5. counting rule
6. final visualization

### Analysis notes

- The threshold value, the ROI placement, and the geometric filtering are the most fragile parts of the crosswalk pipeline.
- The result is sensitive to illumination changes, shadows, perspective, and any bright road markings that survive the threshold.
- Robustness would improve with a better ROI, perspective rectification, stronger shape constraints, and temporal smoothing on video.
